## Setup Libraries

In [1]:
%%time
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 7.4 MB/s eta 0:00:00
CPU times: user 75 ms, sys: 26 ms, total: 101 ms
Wall time: 6.5 s


In [2]:
import os
import random
import warnings
import numpy as np, pandas as pd, seaborn as sns
from colorama import Fore, Style
import itertools
from tqdm.notebook import tqdm
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import tensorflow as tf
import pytabkit
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))

2026-03-28 09:38:29.211551: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774690709.594451      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774690709.703651      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774690710.662546      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774690710.662600      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774690710.662603      24 computation_placer.cc:177] computation placer alr

PyTorch  version: 2.10.0+cu128
PyTabKit version: 1.7.3


In [3]:
## -- Global Settings --
# sklearn.set_config(transform_output="pandas")
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')

# pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', 1000)
sns.set_style("whitegrid")
# plt.style.use("ggplot")

PALETTE = ['#3A86FF', '#F94144', '#FFBE0B', '#73D2DE', '#FBB13C']
sns.set_palette(PALETTE)

cmap = sns.diverging_palette(0, 230, 90, 60, as_cmap=True)

## -- DEfine Configuration --
class CFG:
    FOLDS = 10
    SEED = 42

    GREEN  = '\033[32m'
    YELLOW = '\033[33m'
    RESET  = '\033[0m'

tf.keras.utils.set_random_seed(CFG.SEED)

print(f"CLASSIC {CFG.GREEN} GREEN {CFG.RESET} {CFG.YELLOW} YELLOW {CFG.RESET}")

CLASSIC  GREEN   YELLOW 


In [4]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(seed=CFG.SEED)

## Load Data

In [5]:
PATH = '/kaggle/input/competitions/playground-series-s6e3/'

train = pd.read_csv(PATH+'train.csv')
test = pd.read_csv(PATH+'test.csv')
submit = pd.read_csv(PATH+'sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (594194, 21)
Test shape : (254655, 20)


## Preprocess Features

In [6]:
%%time
ID = 'id'
TARGET = 'Churn'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Contract', 'InternetService', 'PaymentMethod'),
]
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_MonthlyCharges_/_TotalCharges'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1e-6)).astype('float32')
    df['_TotalCharges_/_tenure']  = (df['TotalCharges'] / (df['tenure'] + 1e-6)).astype('float32')
    df['_Monthly_to_avg_ratio'] = (df['MonthlyCharges'] / (df['_TotalCharges_/_tenure'] + 1e-6)).astype('float32')
    df['_TotalCharges_/_MonthlyCharges']  = (df['TotalCharges'] / (df['MonthlyCharges'] + 1e-6)).astype('float32')
    df['_tenure_sq'] = (df['tenure'] ** 2).astype("float32")
    df['is_loyal_customer_'] = (df['tenure'] >= 24).astype('category')

    # for p in [12]:
    #     df[f"_MonthlyCharges_sin_{p}"] = np.sin(2 * np.pi * df['MonthlyCharges'] / p).astype('float32')
    #     df[f"_MonthlyCharges_cos_{p}"] = np.cos(2 * np.pi * df['MonthlyCharges'] / p).astype('float32')
    
    # Digit extraction
    for col in ['TotalCharges']:
        k = -3
        digit_name = f"{col}_d{k}_"
        df[digit_name] = ((df[col] * 10**k) % 10).astype('int8')
        df[digit_name] = df[digit_name].astype('category')

    df['_TotalCharges_mod100'] = (np.floor(df['TotalCharges']) % 100).astype('float32')
    df['_TotalCharges_mod1000'] = (np.floor(df['TotalCharges']) % 1000).astype('float32')
    df['TotalCharges_is_multiple_10_'] = (np.floor(df['TotalCharges']) % 10 == 0).astype('category')

    # Discretize numericals
    bin_config = {'TotalCharges': [4000, 500], 'MonthlyCharges': [200, 100]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_bin_"
            if fit:
                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode='ordinal',
                    strategy='quantile',
                    subsample=None
                )
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype('int32')
            df[bin_name] = binned
            df[bin_name] = df[bin_name].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        round_level = 0
        if fit:
            round_flag = col == 'TotalCharges'
            series = df[col].round(round_level) if round_flag else df[col]
            codes, uniques = series.factorize()
            category_map[col] = {'uniques': uniques, 'round_flag': round_flag}
        else:
            round_flag = category_map[col]['round_flag']
            uniques = category_map[col]['uniques']
            series = df[col].round(round_level) if round_flag else df[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = series.map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes    
        df[cat_name] = df[cat_name].astype('category')

    # Create interaction categories
    combo_names = []
    for col1, col2, col3 in important_combos:
        combo_name = f"{col1}_{col2}_{col3}_"
        combo_names.append(combo_name)
        combo_series = df[col1].astype(str) + '_' + df[col2].astype(str) + '_' + df[col3].astype(str)
        if fit:
            codes, uniques = combo_series.factorize()
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

X      init shape: (594194, 19)
X_test init shape: (254655, 19) 

init len(cat_cols): 15
init len(num_cols): 4 

len(new_cat_cols): 12
len(new_num_cols): 7 

prep len(cat_cols): 27
prep len(num_cols): 11 

X      prep shape: (594194, 38)
X_test prep shape: (254655, 38) 

CPU times: user 1.24 s, sys: 223 ms, total: 1.46 s
Wall time: 1.49 s


## Config

In [7]:
real_params = {
    'random_state': CFG.SEED,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',

    'n_ens': 32,
    'n_epochs': 3,
    'batch_size': 256,
    'use_early_stopping': True,
    'early_stopping_additive_patience': 10,
    'early_stopping_multiplicative_patience': 1,

    'lr': 0.075,
    'wd': 0.0236,
    'sq_mom': 0.988,
    'lr_sched': 'flat_anneal',
    'first_layer_lr_factor': 0.25,
    
    'add_front_scale': False,
    'bias_init_mode': 'neg-uniform-dynamic-2',

    'embedding_size': 6,
    'max_one_hot_cat_size': 18,
    'hidden_sizes': [512, 256, 128],
    'act': 'silu',
    'p_drop': 0.05,
    'p_drop_sched': 'expm4t',

    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_act_name': 'gelu',
    'plr_lr_factor': 0.1151,
    'plr_sigma': 2.33,

    'ls_eps': 0.02,
    'ls_eps_sched': 'sqrt_cos',

    'tfms': ['one_hot', 'median_center', 'robust_scale',
             'smooth_clip', 'embedding', 'l2_normalize'],
}

tab_params = {
    'arch_type': 'tabm-mini-normal',
    'num_emb_type': 'pwl',
    # 'allow_amp': True,
    'lr': 1e-3,
    'batch_size': 256,
    # 'd_block': 512,
    'n_epochs': 5,
    'patience': 3,
    'n_blocks': 3,
    # 'tabm_k': 24,
    'd_embedding': 8,
    'dropout': 0.2,
    # 'gradient_clipping_norm': None,
    # 'num_emb_n_bins': 32,
    # 'weight_decay': 1e-3,
    'verbosity': 2,
    'random_state': CFG.SEED,
    'val_metric_name': '1-auc_ovr',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

## Train K-Fold

In [8]:
%%time
skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print("#"*16)
    print(f"### Fold {fold}/{CFG.FOLDS} ...")
    print("#"*16)

    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Target encoding
    te_cols = combo_names
    TE = TargetEncoder(cv=CFG.FOLDS, smooth='auto', shuffle=True, random_state=CFG.SEED)
    tr_enc = TE.fit_transform(X_tr[te_cols], y.iloc[tr_idx])
    val_enc = TE.transform(X_val[te_cols])
    tst_enc = TE.transform(X_tst[te_cols])

    te_names = [f"_{col}TE" for col in te_cols]
    X_tr[te_names]  = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    model = RealMLP_TD_Classifier(**real_params)
    # model = TabM_D_Classifier(**tab_params)
    model.fit(X_tr, y.iloc[tr_idx], X_val, y.iloc[val_idx], cat_col_names=cat_cols)

    val_preds = model.predict_proba(X_val)[:, 1]
    fold_test_preds = model.predict_proba(X_tst)[:, 1]

    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / CFG.FOLDS

    fold_score = roc_auc_score(y.iloc[val_idx], val_preds)
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} | AUC Score: {fold_score:.5f}{Style.RESET_ALL}\n")
    torch.cuda.empty_cache()

oof_score = str(round(roc_auc_score(y, oof_preds), 5)).split('.')[1]

print("\n" + "="*24)
print(f"Overall OOF AUC: {CFG.GREEN}{oof_score}{CFG.RESET}")
print("="*24)

################
### Fold 1/10 ...
################
Columns classified as continuous: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', '_MonthlyCharges_/_TotalCharges', '_TotalCharges_/_tenure', '_Monthly_to_avg_ratio', '_TotalCharges_/_MonthlyCharges', '_tenure_sq', '_TotalCharges_mod100', '_TotalCharges_mod1000', '_Contract_InternetService_PaymentMethod_TE']
Columns classified as categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'is_loyal_customer_', 'TotalCharges_d-3_', 'TotalCharges_is_multiple_10_', 'TotalCharges_4000_bin_', 'TotalCharges_500_bin_', 'MonthlyCharges_200_bin_', 'MonthlyCharges_100_bin_', 'SeniorCitizen_cat_', 'tenure_cat_', 'MonthlyCharges_cat_', 'TotalCharges_cat_', 'Contract_InternetService_PaymentMethod_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/3: val 1-auc_ovr = 0.081564
Epoch 2/3: val 1-auc_ovr = 0.081220
Epoch 3/3: val 1-auc_ovr = 0.080506


`Trainer.fit` stopped: `max_epochs=3` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 | AUC Score: 0.91949

################
### Fold 2/10 ...
################
Columns classified as continuous: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', '_MonthlyCharges_/_TotalCharges', '_TotalCharges_/_tenure', '_Monthly_to_avg_ratio', '_TotalCharges_/_MonthlyCharges', '_tenure_sq', '_TotalCharges_mod100', '_TotalCharges_mod1000', '_Contract_InternetService_PaymentMethod_TE']
Columns classified as categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'is_loyal_customer_', 'TotalCharges_d-3_', 'TotalCharges_is_multiple_10_', 'TotalCharges_4000_bin_', 'TotalCharges_500_bin_', 'MonthlyCharges_200_bin_', 'MonthlyCharges_100_bin_', 'SeniorCitizen_cat_', 'tenure_cat_', 'MonthlyCharges_cat_', 'TotalCharges_cat_', 'Contract_InternetService_PaymentMethod_']


...

Fold 9 | AUC Score: 0.91832

################
### Fold 10/10 ...
################
Columns classified as continuous: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', '_MonthlyCharges_/_TotalCharges', '_TotalCharges_/_tenure', '_Monthly_to_avg_ratio', '_TotalCharges_/_MonthlyCharges', '_tenure_sq', '_TotalCharges_mod100', '_TotalCharges_mod1000', '_Contract_InternetService_PaymentMethod_TE']
Columns classified as categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'is_loyal_customer_', 'TotalCharges_d-3_', 'TotalCharges_is_multiple_10_', 'TotalCharges_4000_bin_', 'TotalCharges_500_bin_', 'MonthlyCharges_200_bin_', 'MonthlyCharges_100_bin_', 'SeniorCitizen_cat_', 'tenure_cat_', 'MonthlyCharges_cat_', 'TotalCharges_cat_', 'Contract_InternetService_PaymentMethod_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/3: val 1-auc_ovr = 0.083734
Epoch 2/3: val 1-auc_ovr = 0.083458
Epoch 3/3: val 1-auc_ovr = 0.082946


`Trainer.fit` stopped: `max_epochs=3` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 10 | AUC Score: 0.91705


Overall OOF AUC: 91924
CPU times: user 21min 3s, sys: 2min 57s, total: 24min
Wall time: 23min 11s


## Evaluation and Submission

In [9]:
print("\n" + "="*24)
print(f"Overall OOF AUC: {CFG.GREEN}{oof_score}{CFG.RESET}")
print("="*24)

np.save(f"oof_realmlp_{oof_score}.csv", oof_preds)
np.save(f"test_realmlp_{oof_score}.csv", test_preds)

submit[TARGET] = test_preds
submit.to_csv(f"submit_realmlp_{oof_score}.csv", index=False)
submit.head()


Overall OOF AUC: 91924


,id,Churn
0,594194,0.123768
1,594195,0.001640
2,594196,0.115354
3,594197,0.003910
4,594198,0.519645


In [10]:
# !rm -r /kaggle/working